# Data Augmentation

In [2]:
import pandas as pd
import numpy as np
import requests
import time
from io import StringIO

In [3]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
ORIGINAL_TRAIN_FILE = '../data/train.csv'
OUTPUT_FILE = '../data/train_augmented_v2.csv'

# External Data Files
EXT_FILES = {
    'S-glutathionylation': '../data/Glutathionylation_dbPTM.csv',
    'S-nitrosylation':     '../data/S-nitrosylation_dbPTM.csv',
    'S-palmitoylation':    '../data/S-palmitoylation_dbPTM.csv'
}

In [4]:
# ==============================================================================
# 1. PARSE EXTERNAL DATA
# ==============================================================================
print("Parsing external files...")
records = []

for label, filepath in EXT_FILES.items():
    # Assuming no header based on your snippet
    # Cols: 0=Name, 1=UniProtID, 2=Position, 3=Type, 4=PubMed, 5=21mer
    try:
        df = pd.read_csv(filepath, header=None)
        
        for _, row in df.iterrows():
            records.append({
                'ID': row[1],           # UniProt ID (e.g., O00299)
                'Position': int(row[2]), # 1-based index
                'Label': label
            })
        print(f"  Loaded {len(df)} entries from {label}")
    except FileNotFoundError:
        print(f"  WARNING: File not found: {filepath}")

df_ext = pd.DataFrame(records)
print(f"Total external sites found: {len(df_ext)}")

Parsing external files...
  Loaded 4129 entries from S-glutathionylation
  Loaded 4172 entries from S-nitrosylation
  Loaded 6501 entries from S-palmitoylation
Total external sites found: 14802


In [5]:
# ==============================================================================
# 2. BATCH FETCH FULL SEQUENCES FROM UNIPROT
# ==============================================================================
unique_ids = df_ext['ID'].unique()
print(f"\nFetching sequences for {len(unique_ids)} unique proteins from UniProt...")

def fetch_uniprot_sequences(id_list):
    """
    Fetches full sequences in batch from UniProt API.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/accessions"
    
    # UniProt accepts batch requests, but let's chunk it to be safe
    chunk_size = 500 
    id_to_seq = {}
    
    for i in range(0, len(id_list), chunk_size):
        chunk = id_list[i:i+chunk_size]
        params = {
            'accessions': ','.join(chunk),
            'format': 'fasta'
        }
        
        response = requests.get(base_url, params=params)
        
        if response.status_code == 200:
            # Simple FASTA parser
            current_id = None
            current_seq = []
            
            for line in response.text.splitlines():
                if line.startswith('>'):
                    # Save previous
                    if current_id:
                        id_to_seq[current_id] = ''.join(current_seq)
                    
                    # Parse new ID (e.g., >sp|O00299|...)
                    # Extract the ID between | and |
                    parts = line.split('|')
                    if len(parts) >= 2:
                        current_id = parts[1]
                    current_seq = []
                else:
                    current_seq.append(line.strip())
            
            # Save last entry
            if current_id:
                id_to_seq[current_id] = ''.join(current_seq)
        
        print(f"  Fetched {min(i+chunk_size, len(id_list))}/{len(id_list)}")
        time.sleep(1) # Be polite to the API
        
    return id_to_seq

# Execute Fetch
protein_sequences = fetch_uniprot_sequences(list(unique_ids))
print(f"Successfully retrieved {len(protein_sequences)} sequences.")


Fetching sequences for 6493 unique proteins from UniProt...
  Fetched 500/6493
  Fetched 1000/6493
  Fetched 1500/6493
  Fetched 2000/6493
  Fetched 2500/6493
  Fetched 3000/6493
  Fetched 3500/6493
  Fetched 4000/6493
  Fetched 4500/6493
  Fetched 5000/6493
  Fetched 5500/6493
  Fetched 6000/6493
  Fetched 6493/6493
Successfully retrieved 6315 sequences.


In [6]:
# ==============================================================================
# 3. EXTRACT 31-MERS AND HANDLE MULTI-LABELS
# ==============================================================================
print("\nExtracting 31-mers and merging labels...")

processed_data = {} # Key: (ID, Sequence_31mer) -> Value: Label Dict

for _, row in df_ext.iterrows():
    uid = row['ID']
    pos_1based = row['Position']
    label_type = row['Label']
    
    if uid not in protein_sequences:
        continue
        
    full_seq = protein_sequences[uid]
    
    # Convert 1-based index to 0-based
    center_idx = pos_1based - 1
    
    # Verify it is Cysteine (optional but recommended safety check)
    if center_idx < 0 or center_idx >= len(full_seq):
        continue # Out of bounds
        
    # Extract 31-mer (Center +/- 15)
    start = center_idx - 15
    end = center_idx + 16
    
    # Handle physical boundaries (This is REAL padding, not artificial 'X' padding)
    # If the window goes outside the protein, pad with 'X' only then.
    seq_fragment = ""
    
    # Left padding
    if start < 0:
        seq_fragment += "X" * abs(start)
        valid_start = 0
    else:
        valid_start = start
        
    # Sequence content
    seq_fragment += full_seq[valid_start:end]
    
    # Right padding
    if end > len(full_seq):
        seq_fragment += "X" * (end - len(full_seq))
        
    # Final length check
    if len(seq_fragment) != 31:
        continue
        
    # Aggregate Labels (Handle Multi-label PTMs)
    key = (uid, seq_fragment)
    
    if key not in processed_data:
        processed_data[key] = {
            'S-glutathionylation': 0,
            'S-nitrosylation': 0,
            'S-palmitoylation': 0
        }
    
    # Mark the specific label as active
    processed_data[key][label_type] = 1

# Convert to DataFrame
final_rows = []
for (uid, seq), labels in processed_data.items():
    row = {
        'ID': uid,
        'Sequence': seq,
        'S-glutathionylation': labels['S-glutathionylation'],
        'S-nitrosylation': labels['S-nitrosylation'],
        'S-palmitoylation': labels['S-palmitoylation']
    }
    final_rows.append(row)

df_augmented = pd.DataFrame(final_rows)
print(f"Generated {len(df_augmented)} unique 31-mer positive samples.")


Extracting 31-mers and merging labels...
Generated 12249 unique 31-mer positive samples.


In [7]:
# ==============================================================================
# 4. MERGE WITH ORIGINAL TRAIN.CSV
# ==============================================================================
print("\nMerging with original training data...")
df_train = pd.read_csv(ORIGINAL_TRAIN_FILE)

# Concatenate
df_final = pd.concat([df_train, df_augmented], axis=0)

# Drop duplicates (if any external data overlaps with provided training data)
# We keep the one with more positive labels if there's a conflict
df_final = df_final.groupby(['ID', 'Sequence'], as_index=False).max()

# Shuffle
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
df_final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Saved augmented dataset to: {OUTPUT_FILE}")
print(f"  Original size: {len(df_train)}")
print(f"  New size:      {len(df_final)}")


Merging with original training data...

✓ Saved augmented dataset to: ../data/train_augmented_v2.csv
  Original size: 89010
  New size:      97030
